# Vintage-aware cost reconstruction — 12 runs (5x2025, 3x2035, 4x2050)

The AMPL model's own `C_inv` excludes `f_min` capacity from investment cost entirely (`C_inv[c,j] = c_inv[c,j] * (F[c,j] - f_min[c,j])`, `ESMC_model_AMPL.mod:340`) -- so for a 2050 run, capacity chained forward from a 2035 solve (`f_min[2050]`) is implicitly free in the LP's own `TotalCost`. That is correct for the LP's own decision problem (that capacity is sunk, already paid for in the 2035 run), but it is not a fair like-for-like system cost: a 2050 run with a lot of inherited 2035 capacity looks artificially cheap next to one that had to build everything fresh at 2050 prices.

This notebook reconstructs a vintage-aware total cost: on top of the LP's own `TotalCost`, it adds back the annualized cost of every `chained_to_2050` capacity tranche in `output_energyscope_2050/vintage_registry.csv`, priced at **that tranche's own vintage-year `c_inv`** (2035, ratio-adjusted where applicable) -- never at the 2050 catalog's price, and never re-added for `retired_at_2050` rows (already amortized in the 2035 run's own total, not double-counted here). 2025 and 2035 runs have no such adjustment (2035 is the first horizon capacity is chained *into*, not *out of* a prior solved run), so LP raw and reconstructed are identical for those 10 runs by construction.

## Published cost metric: reconstructed, not LP raw (decision, 2026-08-06)

**The reconstructed cost is the headline number reported throughout the memoire for every scenario and horizon. LP raw is kept only as a secondary/technical reference column, never presented as a competing headline figure.** This is a single global decision, not made scenario by scenario: a capacity tranche built in 2035 is still being amortized in 2050 (its lifetime has not elapsed), so its cost belongs in a 2050 system-cost comparison, unlike the 2025 fleet, which really is sunk by 2050 for every technology chained here (lifetimes involved are all under 30y). Reporting LP raw as the headline number would understate the true cost of scenarios that inherit a lot of 2035 capacity relative to scenarios that have to build fresh at 2050 prices -- exactly backwards from what a fair comparison needs.

**This choice changes the ranking, not just the level**: at 2050, `early_access` costs less than `late_access` under LP raw (36.90 vs 38.36 M€/y) but *more* under reconstructed (53.44 vs 53.06 M€/y) -- see the table below. Any statement in the memoire comparing scenarios at 2050 must use the reconstructed column; citing LP raw for that comparison would silently flip which scenario looks cheaper.

In [1]:
import pandas as pd
from pathlib import Path

ES_ROOT = Path(r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
CS_ROOT = ES_ROOT / "case_studies" / "C1_C2_C3_C4_C5"
REGISTRY_PATH = Path("output_energyscope_2050/vintage_registry.csv")

# i_rate is a project-wide constant (Data/*/00_INDEP -> indep.dat), confirmed 0.1 in every horizon
I_RATE = 0.1

RUNS = [
    # (scenario label, year, case_study folder, Data/<year>/<scenario> for lifetime lookup, is_2050)
    ("reality",             2025, "norte_amazonia_reality_2025",             "reality",             False),
    ("reality_phase2",      2025, "norte_amazonia_reality_2025_phase2",      "reality",             False),
    ("reality_access",      2025, "norte_amazonia_reality_access_2025",      "reality_access",      False),
    ("access_transition",   2025, "norte_amazonia_access_transition_2025",   "reality_access",      False),
    ("sufficiency",         2025, "norte_amazonia_sufficiency_2025",         "sufficiency",         False),
    ("early_access",        2035, "norte_amazonia_early_access_2035",        "early_access",        False),
    ("late_access",         2035, "norte_amazonia_late_access_2035",         "late_access",         False),
    ("no_transition",       2035, "norte_amazonia_no_transition_2035",       "no_transition",       False),
    ("early_access",        2050, "norte_amazonia_early_access_2050",        "early_access",        True),
    ("early_access_brazil", 2050, "norte_amazonia_early_access_brazil_2050", "early_access_brazil", True),
    ("late_access",         2050, "norte_amazonia_late_access_2050",         "late_access",         True),
    ("no_transition",       2050, "norte_amazonia_no_transition_2050",       "no_transition",       True),
]

registry = pd.read_csv(REGISTRY_PATH, sep=";")
print(f"Registry loaded: {len(registry)} rows")

Registry loaded: 596 rows


In [2]:
rows = []
for label, year, case_study, data_scenario, is_2050 in RUNS:
    out_dir = CS_ROOT / case_study / "outputs"

    solve_info = pd.read_csv(out_dir / "Solve_info.csv", sep=r"\t;\t", header=None,
                              index_col=0, engine="python")
    solve_result_num = int(float(solve_info.loc["solve_result_num", 1]))
    assert solve_result_num == 0, f"{case_study}: solve_result_num={solve_result_num} != 0 -- refusing to use this run"

    total_cost_lp = pd.read_csv(out_dir / "TotalCost.csv", sep=";", index_col=0).iloc[0, 0]
    cost_breakdown = pd.read_csv(out_dir / "Cost_breakdown.csv", sep=";", index_col=0).fillna(0.0)

    # lifetime + category catalog for this run's deployed technologies (needed for tau + storage tagging)
    cat_path = ES_ROOT / "Data" / str(year) / data_scenario / "02_REF_REGION" / "Technologies.csv"
    catalog = pd.read_csv(cat_path, sep=";", skiprows=[1])
    catalog["Technologies param"] = catalog["Technologies param"].str.strip()
    catalog = catalog.set_index("Technologies param")

    reconstructed_c_inv = cost_breakdown["C_inv"].copy()
    adjustment_total = 0.0

    if is_2050:
        chained = registry[(registry["scenario"] == label) & (registry["status"] == "chained_to_2050")]
        for _, r in chained.iterrows():
            tech = r["technology"]
            if tech not in catalog.index:
                continue
            lifetime = float(catalog.loc[tech, "lifetime"])
            tau = I_RATE * (1 + I_RATE) ** lifetime / ((1 + I_RATE) ** lifetime - 1)
            addon = r["capacity_GW"] * r["c_inv"] * tau
            if tech not in reconstructed_c_inv.index:
                reconstructed_c_inv[tech] = 0.0
            reconstructed_c_inv[tech] += addon
            adjustment_total += addon

    reconstructed_breakdown = cost_breakdown.copy()
    reconstructed_breakdown["C_inv"] = reconstructed_c_inv
    reconstructed_total = float(reconstructed_breakdown[["C_inv", "C_maint", "C_op"]].sum().sum())

    # sanity: reconstructed total = LP total + adjustment, up to the pre-existing
    # Cost_breakdown.csv-vs-TotalCost.csv rounding residual (~1e-5, present even at zero
    # adjustment -- a CSV-rounding artifact, not something introduced by this reconstruction)
    residual = reconstructed_total - (total_cost_lp + adjustment_total)
    assert abs(residual) < 1e-3, (
        f"{case_study}: reconstructed total {reconstructed_total} != LP {total_cost_lp} + "
        f"adjustment {adjustment_total} (residual {residual})")

    storage_techs = [t for t in catalog.index if str(catalog.loc[t, "Category"]).strip() == "Storage"
                      and t in reconstructed_breakdown.index]
    storage_cost = float(reconstructed_breakdown.loc[storage_techs, ["C_inv", "C_maint", "C_op"]].sum().sum())
    storage_share = storage_cost / reconstructed_total if reconstructed_total else float("nan")

    rows.append(dict(
        year=year, scenario=label, case_study=case_study,
        # published headline metric first -- see the scope-declaration cell above
        reconstructed_cost_Meur=reconstructed_total,
        lp_raw_cost_Meur_secondary=total_cost_lp,
        vintage_adjustment_Meur=adjustment_total,
        reconstructed_vs_lp_pct=(reconstructed_total - total_cost_lp) / total_cost_lp * 100 if total_cost_lp else float("nan"),
        storage_share_pct=storage_share * 100,
    ))

table = pd.DataFrame(rows)
pd.set_option("display.width", 160)
print(table.to_string(index=False))
print()
print("=== Ranking check: does early_access vs late_access flip between LP raw and reconstructed at 2050? ===")
t2050 = table[table["year"] == 2050].set_index("scenario")
ea_lp, la_lp = t2050.loc["early_access", "lp_raw_cost_Meur_secondary"], t2050.loc["late_access", "lp_raw_cost_Meur_secondary"]
ea_rc, la_rc = t2050.loc["early_access", "reconstructed_cost_Meur"], t2050.loc["late_access", "reconstructed_cost_Meur"]
print(f"LP raw:        early_access={ea_lp:.2f}  late_access={la_lp:.2f}  -> {'early cheaper' if ea_lp < la_lp else 'late cheaper'}")
print(f"Reconstructed: early_access={ea_rc:.2f}  late_access={la_rc:.2f}  -> {'early cheaper' if ea_rc < la_rc else 'late cheaper'}")

 year            scenario                              case_study  reconstructed_cost_Meur  lp_raw_cost_Meur_secondary  vintage_adjustment_Meur  reconstructed_vs_lp_pct  storage_share_pct
 2025             reality             norte_amazonia_reality_2025                66.647833                   66.647840                 0.000000                -0.000011           0.031613
 2025      reality_phase2      norte_amazonia_reality_2025_phase2                42.285620                   42.285628                 0.000000                -0.000021          30.251466
 2025      reality_access      norte_amazonia_reality_access_2025                69.733770                   69.733779                 0.000000                -0.000012           0.213828
 2025   access_transition   norte_amazonia_access_transition_2025                47.739853                   47.739862                 0.000000                -0.000020          28.877768
 2025         sufficiency         norte_amazonia_sufficiency

## Save the table

**Caveat -- read before interpreting this table**: this reconstruction was built after the 3x2035 + 4x2050 reruns under the NREL ATB ratios had already overwritten the pre-ratio `case_studies/` outputs. The pre-ratio `TotalCost.csv` values were not snapshotted beforehand, so this notebook can only report the *post-ratio* LP-raw and reconstructed costs for all 12 runs -- it cannot compute a numeric before/after delta for the 7 projection scenarios (only the 5x2025 runs are a valid before/after check, since they were never touched by any ratio: their LP raw cost here is byte-for-byte the pre-existing solved value). Any claim about "costs falling" for the 2035/2050 scenarios versus their pre-ratio baseline would need those scenarios re-solved under the old flat catalog for a genuine comparison.

In [3]:
OUT_PATH = "output_energyscope_2050/cost_reconstruction_table.csv"
table.to_csv(OUT_PATH, sep=";", index=False)
print(f"Saved to {OUT_PATH}")

Saved to output_energyscope_2050/cost_reconstruction_table.csv
